# 01. W&B 처음 써 보기 (YOLO 없이)

목적: **W&B 구조(project / run / config / log)만** 이해하기.

진짜 학습 대신, epoch마다 줄어드는 **가짜 loss**를 만들어서 W&B로 보낸다.

| 개념 | 코드 | 뜻 |
|---|---|---|
| project | `wandb.init(project=...)` | 실험들을 모아 두는 폴더 |
| run | `wandb.init()`이 돌려주는 `run` | 학습 한 번 |
| config | `wandb.init(config=...)` | 그 run의 설정값 |
| log | `run.log({...})` | 숫자를 보냄 → 웹에서 그래프에 점이 찍힘 |
| finish | `run.finish()` | run 종료 |

> 사전 준비: 터미널에서 `pip install wandb` → `wandb login` (완료)

## 1. 라이브러리 불러오기

In [1]:
import math
import random

import wandb

print(wandb.__version__)

PROJECT = "wandb-study"  # 웹에서 보일 project 이름

0.30.0


## 2. run 시작하기

- `project`: 이 run을 넣을 폴더 이름. 없으면 자동으로 만들어진다.
- `name`: 웹에서 보일 run 이름. 안 적으면 `happy-sun-3` 같은 랜덤 이름이 붙는다.
- `config`: 이번 run의 설정값. 웹의 Overview 탭에 그대로 저장된다.

실행하면 아래에 **웹 링크**가 출력된다. 눌러서 열어 두자.

In [2]:
run = wandb.init(
    project=PROJECT,
    name="first-run",
    config={"lr": 0.1, "epochs": 20},
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\hurwa\_netrc.
wandb: Currently logged in as: hurwan0629 (hurwan0629-something) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. 가짜 학습 루프에서 숫자 보내기

- `run.config.lr`: 위에서 넣은 설정값을 다시 꺼내 쓴다. (설정을 한 곳에서만 관리)
- 가짜 loss = `exp(-lr × epoch)` + 약간의 노이즈 → epoch이 갈수록 줄어드는 곡선
- `run.log({...})`: 한 번 부를 때마다 그래프에 점이 하나씩 찍힌다. (`wandb.log`와 같은 기능)

실행하면서 웹 페이지를 보면 그래프가 그려지는 걸 볼 수 있다.

In [3]:
for epoch in range(run.config.epochs):
    loss = math.exp(-run.config.lr * epoch) + random.uniform(0, 0.05)
    run.log({"epoch": epoch, "loss": loss})

print("마지막 loss:", loss)

마지막 loss: 0.15903317193388866


## 4. run 끝내기

`finish()`를 해야 기록이 서버에 다 올라가고 run이 "Finished" 상태가 된다.

안 하면 다음 `wandb.init()`과 섞이거나 run이 "Running"으로 남는다.

In [4]:
run.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
loss,█▇▆▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁
epoch,19
loss,0.15903


## 5. 설정만 바꿔서 run 2개 더 돌리고 비교하기

2~4번(init → log → finish)을 lr만 바꿔서 반복한다.

끝나면 웹에서 project 페이지(`wandb-study`)로 가서 **loss 그래프에 run 3개가 겹쳐 보이는지** 확인한다.

→ lr이 클수록 loss가 빨리 줄어드는 게 보여야 한다.

In [5]:
for lr in [0.05, 0.3]:
    run = wandb.init(
        project=PROJECT,
        name=f"lr-{lr}",
        config={"lr": lr, "epochs": 20},
    )

    for epoch in range(run.config.epochs):
        loss = math.exp(-run.config.lr * epoch) + random.uniform(0, 0.05)
        run.log({"epoch": epoch, "loss": loss})

    run.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
loss,█▇▇▆▆▅▅▅▄▄▃▃▃▃▂▂▂▁▁▁
epoch,19
loss,0.39521


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
loss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,19
loss,0.01246


## 6. 웹에서 확인할 것

- [x] project 페이지에 run 3개(`first-run`, `lr-0.05`, `lr-0.3`)가 보이는가
- [x] loss 그래프에 3개 곡선이 겹쳐 보이는가
- [x] 왼쪽 run 목록(Runs 표)에 `lr` 열이 있는가 → config가 표의 열이 된다
- [x] run 하나를 눌러서 Overview 탭에 config가 저장돼 있는가
- [x] 이 폴더에 `wandb/` 폴더가 생겼는가 → 로컬 기록(캐시). git에는 안 올라감